# 🧠 신경망 한 층의 순전파(Forward Pass) — 인터랙티브 시각화

> **4주차 보강 자료** | NumPy가 곧 신경망이다

---

## 이 노트북에서 배우는 것

| 단계 | 내용 | 핵심 연결 |
|:---:|---|---|
| ① | **입력**(Input) → **가중치**(Weight) 행렬 곱 | NumPy의 `@` 연산자 |
| ② | **편향**(Bias) 덧셈 | 브로드캐스팅(Broadcasting) |
| ③ | **활성화 함수**(Activation Function) 적용 | 비선형성 — 왜 필요한가? |
| ④ | **전체 순전파** 시각화 | 이것이 GPT의 기본 단위이다 |

---

### 🔑 핵심 메시지

```
a = activation(X @ W + b)
```

**NumPy 한 줄이면 신경망 한 층이 완성된다.** GPT, BERT, Stable Diffusion 모두 이 연산의 반복이다.


## 0. 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 (Colab) ──
import os
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# ── 색상 팔레트 (색맹 안전) ──
C = {
    'input':  '#3b82f6',  # 파랑
    'weight': '#f59e0b',  # 노랑/주황
    'bias':   '#8b5cf6',  # 보라
    'sum':    '#ef4444',  # 빨강
    'act':    '#10b981',  # 초록
    'output': '#10b981',  # 초록
    'bg':     '#0f172a',  # 배경
    'card':   '#1e293b',  # 카드
    'text':   '#e2e8f0',  # 텍스트
    'dim':    '#94a3b8',  # 흐린 텍스트
    'muted':  '#64748b',  # 더 흐린 텍스트
    'accent': '#60a5fa',  # 강조
}

print("✅ 환경 설정 완료!")
print(f"   NumPy 버전: {np.__version__}")
print(f"   신경망 시각화 준비 완료")


---

## 1. 뉴런(Neuron) 하나가 하는 일

신경망의 가장 작은 단위인 **뉴런**(Neuron)은 세 가지를 수행한다:

```
① 입력(x)에 가중치(w)를 곱한다     →  x × w
② 편향(b)을 더한다                 →  x × w + b
③ 활성화 함수를 통과시킨다           →  f(x × w + b)
```

이것을 NumPy로 구현하면 **단 한 줄**이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 뉴런 하나의 동작을 시각화
# ═══════════════════════════════════════════════════

def visualize_single_neuron(x=1.5, w=0.7, b=0.2):
    """뉴런 하나의 계산 과정을 단계별로 시각화한다."""

    z = x * w + b                          # 가중합
    a_relu = max(0, z)                     # ReLU
    a_sigmoid = 1 / (1 + np.exp(-z))       # Sigmoid

    fig, axes = plt.subplots(1, 3, figsize=(15, 4),
                              facecolor=C['bg'])

    steps = [
        ("① 곱셈: x × w", x, '*', w, '=', x*w,
         C['input'], C['weight'], "입력 × 가중치"),
        ("② 덧셈: (x×w) + b", x*w, '+', b, '=', z,
         C['weight'], C['bias'], "가중합 + 편향"),
        ("③ 활성화: f(z)", z, '→', None, None, None,
         C['sum'], C['act'], "비선형 변환"),
    ]

    for idx, (title, v1, op, v2, eq, result, c1, c2, desc) in enumerate(steps):
        ax = axes[idx]
        ax.set_facecolor(C['card'])
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 6)
        ax.axis('off')
        ax.set_title(title, color=C['text'], fontsize=13, fontweight='bold', pad=12)

        if idx < 2:  # 곱셈, 덧셈
            # 왼쪽 값
            circle1 = plt.Circle((2.5, 3), 0.9, color=c1, alpha=0.2)
            ax.add_patch(circle1)
            ax.add_patch(plt.Circle((2.5, 3), 0.9, fill=False,
                                     edgecolor=c1, linewidth=2))
            ax.text(2.5, 3, f'{v1:.2f}', ha='center', va='center',
                    color=c1, fontsize=14, fontweight='bold',
                    fontfamily='monospace')

            # 연산자
            ax.text(4.2, 3, op, ha='center', va='center',
                    color=C['dim'], fontsize=20, fontweight='bold')

            # 오른쪽 값
            circle2 = plt.Circle((5.8, 3), 0.9, color=c2, alpha=0.2)
            ax.add_patch(circle2)
            ax.add_patch(plt.Circle((5.8, 3), 0.9, fill=False,
                                     edgecolor=c2, linewidth=2))
            ax.text(5.8, 3, f'{v2:.2f}', ha='center', va='center',
                    color=c2, fontsize=14, fontweight='bold',
                    fontfamily='monospace')

            # 등호 + 결과
            ax.text(7.2, 3, '=', ha='center', va='center',
                    color=C['dim'], fontsize=20)

            box = FancyBboxPatch((7.8, 2.2), 1.8, 1.6, boxstyle="round,pad=0.2",
                                  facecolor=C['bg'], edgecolor=C['accent'],
                                  linewidth=2)
            ax.add_patch(box)
            ax.text(8.7, 3, f'{result:.2f}', ha='center', va='center',
                    color=C['accent'], fontsize=15, fontweight='bold',
                    fontfamily='monospace')

        else:  # 활성화 함수
            # z 값
            ax.text(1.5, 3, f'z = {z:.2f}', ha='center', va='center',
                    color=C['sum'], fontsize=13, fontweight='bold',
                    fontfamily='monospace')

            # 화살표
            ax.annotate('', xy=(4, 3), xytext=(2.8, 3),
                       arrowprops=dict(arrowstyle='->', color=C['dim'],
                                      lw=2))

            # ReLU 결과
            box_relu = FancyBboxPatch((4.2, 3.5), 2.5, 1.5,
                                       boxstyle="round,pad=0.2",
                                       facecolor=C['act']+'22',
                                       edgecolor=C['act'], linewidth=2)
            ax.add_patch(box_relu)
            ax.text(5.45, 4.8, 'ReLU', ha='center', va='center',
                    color=C['act'], fontsize=10, fontweight='bold')
            ax.text(5.45, 3.9, f'{a_relu:.2f}', ha='center', va='center',
                    color=C['act'], fontsize=14, fontweight='bold',
                    fontfamily='monospace')

            # Sigmoid 결과
            box_sig = FancyBboxPatch((4.2, 1), 2.5, 1.5,
                                      boxstyle="round,pad=0.2",
                                      facecolor=C['accent']+'22',
                                      edgecolor=C['accent'], linewidth=2)
            ax.add_patch(box_sig)
            ax.text(5.45, 2.3, 'Sigmoid', ha='center', va='center',
                    color=C['accent'], fontsize=10, fontweight='bold')
            ax.text(5.45, 1.4, f'{a_sigmoid:.4f}', ha='center', va='center',
                    color=C['accent'], fontsize=14, fontweight='bold',
                    fontfamily='monospace')

        # 하단 설명
        ax.text(5, 0.3, desc, ha='center', va='center',
                color=C['muted'], fontsize=10)

    plt.tight_layout()
    plt.show()

    # NumPy 코드 출력
    print(f"\n{'─'*55}")
    print(f"  🐍 NumPy 코드:")
    print(f"{'─'*55}")
    print(f"  x = {x}")
    print(f"  w = {w}")
    print(f"  b = {b}")
    print(f"  z = x * w + b          # = {z:.4f}")
    print(f"  a = max(0, z)           # ReLU  = {a_relu:.4f}")
    print(f"  a = 1/(1+np.exp(-z))    # Sigmoid = {a_sigmoid:.4f}")
    print(f"{'─'*55}")

visualize_single_neuron(x=1.5, w=0.7, b=0.2)


---

## 2. 활성화 함수(Activation Function) — 왜 필요한가?

활성화 함수가 없으면 아무리 층을 쌓아도 **선형 변환의 반복**에 불과하다.

```
층1: y = W₁x + b₁
층2: y = W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂)
```

이것은 결국 `y = W'x + b'` — **하나의 직선**이다. 곡선, 원, 복잡한 패턴은 절대 표현할 수 없다.

**활성화 함수가 "꺾임"을 만들어야** 비로소 신경망이 복잡한 패턴을 학습할 수 있다.


In [ ]:
# ═══════════════════════════════════════════════════
# 활성화 함수 4종 비교 시각화
# ═══════════════════════════════════════════════════

z = np.linspace(-5, 5, 500)

activations = {
    'ReLU':    (np.maximum(0, z),       C['act'],    'max(0, z)',
                '가장 널리 사용. 음수→0, 양수→그대로'),
    'Sigmoid': (1/(1+np.exp(-z)),       C['accent'], '1/(1+e⁻ᶻ)',
                '출력을 0~1 확률로 변환. 로지스틱 회귀의 핵심'),
    'Tanh':    (np.tanh(z),             C['weight'], '(eᶻ-e⁻ᶻ)/(eᶻ+e⁻ᶻ)',
                '출력 -1~1. LSTM/GRU에서 사용'),
    'Leaky ReLU': (np.where(z > 0, z, 0.1*z), C['bias'], 'max(0.1z, z)',
                   'ReLU 개선. 음수에서도 약간의 기울기 유지'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=C['bg'])
axes = axes.flatten()

for idx, (name, (values, color, formula, desc)) in enumerate(activations.items()):
    ax = axes[idx]
    ax.set_facecolor(C['card'])

    # 기준선
    ax.axhline(y=0, color=C['muted'], linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color=C['muted'], linewidth=0.5, alpha=0.5)

    # 함수 그래프
    ax.plot(z, values, color=color, linewidth=3, label=name)

    # 선형 함수 비교 (점선)
    ax.plot(z, z, color=C['muted'], linewidth=1, linestyle='--',
            alpha=0.4, label='선형 (y=z)')

    # 제목 + 수식
    ax.set_title(f'{name}', color=color, fontsize=15,
                 fontweight='bold', pad=12)
    ax.text(0.05, 0.92, f'f(z) = {formula}', transform=ax.transAxes,
            color=color, fontsize=11, fontfamily='monospace',
            fontweight='bold', alpha=0.9)
    ax.text(0.05, 0.82, desc, transform=ax.transAxes,
            color=C['dim'], fontsize=9)

    # 스타일
    ax.set_xlabel('z (가중합)', color=C['dim'], fontsize=10)
    ax.set_ylabel('f(z) (출력)', color=C['dim'], fontsize=10)
    ax.tick_params(colors=C['muted'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(C['muted'])
        spine.set_alpha(0.3)
    ax.set_xlim(-5, 5)
    ax.legend(loc='lower right', fontsize=9,
              facecolor=C['bg'], edgecolor=C['muted'],
              labelcolor=C['dim'])

plt.suptitle('활성화 함수(Activation Function) 비교',
             color=C['text'], fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 핵심: 활성화 함수가 '꺾임'을 만들어야 신경망이 복잡한 패턴을 학습할 수 있다.")
print("   ReLU가 가장 많이 쓰이는 이유: 계산이 빠르고, 기울기 소실(Vanishing Gradient) 문제가 적다.")
print("   → GPT, BERT 등 최신 모델은 ReLU의 변형인 GELU(Gaussian Error Linear Unit)를 사용한다.")


---

## 3. 신경망 한 층(Layer)의 순전파 — 전체 과정 시각화

뉴런 하나가 아닌, **한 층 전체**를 한 번에 계산하는 것이 핵심이다.

```
입력 3개 → 출력 2개   (가중치 3×2 = 6개, 편향 2개, 총 파라미터 8개)
```

### 행렬 곱셈으로 한 번에 계산

```python
z = X @ W + b        # @ = 행렬 곱(dot product)
a = relu(z)          # 활성화 함수 적용
```

`for`문 없이 **모든 뉴런의 계산이 동시에** 수행된다. 이것이 GPU 가속의 본질이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 신경망 한 층 — 정적 시각화 (전체 구조)
# ═══════════════════════════════════════════════════

def draw_network_layer(X, W, b, activation='relu', figsize=(16, 10)):
    """
    신경망 한 층의 순전파 전체 과정을 시각화한다.

    Parameters
    ----------
    X : np.ndarray, shape (n_input,)
    W : np.ndarray, shape (n_input, n_output)
    b : np.ndarray, shape (n_output,)
    activation : str, 'relu' or 'sigmoid'
    """
    n_in = len(X)
    n_out = len(b)

    # ── 순전파 계산 ──
    z = X @ W + b
    if activation == 'relu':
        a = np.maximum(0, z)
        act_name = 'ReLU'
    else:
        a = 1 / (1 + np.exp(-z))
        act_name = 'Sigmoid'

    fig, ax = plt.subplots(figsize=figsize, facecolor=C['bg'])
    ax.set_facecolor(C['bg'])
    ax.set_xlim(-1, 11)
    ax.set_ylim(-1, max(n_in, n_out) * 2 + 1)
    ax.axis('off')
    ax.set_aspect('equal')

    # 층 x좌표
    layer_x = {'input': 1, 'sum': 4.5, 'act': 7, 'output': 9.5}

    # y좌표 계산
    def get_ys(n, total_height=None):
        if total_height is None:
            total_height = max(n_in, n_out) * 2
        spacing = total_height / (n + 1)
        return [total_height - spacing * (i + 1) for i in range(n)]

    total_h = max(n_in, n_out) * 2
    in_ys = get_ys(n_in, total_h)
    out_ys = get_ys(n_out, total_h)

    # ── 연결선 (가중치) ──
    for i, iy in enumerate(in_ys):
        for j, oy in enumerate(out_ys):
            w_val = W[i, j]
            thickness = max(0.5, min(4, abs(w_val) * 3))
            alpha_v = max(0.2, min(0.8, abs(w_val)))
            color = C['weight'] if w_val >= 0 else C['sum']

            ax.plot([layer_x['input'] + 0.45, layer_x['sum'] - 0.45],
                    [iy, oy],
                    color=color, linewidth=thickness, alpha=alpha_v,
                    zorder=1)

            # 가중치 라벨
            mx = (layer_x['input'] + layer_x['sum']) / 2
            my = (iy + oy) / 2
            ax.text(mx, my + 0.15, f'w={w_val:.1f}',
                    ha='center', va='center', fontsize=7,
                    color=C['weight'], fontfamily='monospace',
                    alpha=0.8,
                    bbox=dict(boxstyle='round,pad=0.15',
                             facecolor=C['bg'], edgecolor=C['weight'],
                             alpha=0.7, linewidth=0.5))

    # ── 합산 → 활성화 → 출력 연결선 ──
    for j, oy in enumerate(out_ys):
        # sum → activation
        ax.annotate('', xy=(layer_x['act'] - 0.45, oy),
                   xytext=(layer_x['sum'] + 0.45, oy),
                   arrowprops=dict(arrowstyle='->', color=C['act'],
                                  lw=1.5, alpha=0.6))
        # activation → output
        ax.annotate('', xy=(layer_x['output'] - 0.45, oy),
                   xytext=(layer_x['act'] + 0.45, oy),
                   arrowprops=dict(arrowstyle='->', color=C['output'],
                                  lw=2, alpha=0.7))

    # ── 뉴런 그리기 함수 ──
    def draw_neuron(x, y, value, color, label_above='', label_below='',
                    radius=0.4, fontsize=11):
        circle_bg = plt.Circle((x, y), radius, color=color, alpha=0.15, zorder=3)
        circle_bd = plt.Circle((x, y), radius, fill=False,
                                edgecolor=color, linewidth=2.5, zorder=4)
        ax.add_patch(circle_bg)
        ax.add_patch(circle_bd)

        if isinstance(value, float):
            txt = f'{value:.2f}'
        else:
            txt = str(value)
        ax.text(x, y, txt, ha='center', va='center',
                color=color, fontsize=fontsize, fontweight='bold',
                fontfamily='monospace', zorder=5)

        if label_above:
            ax.text(x, y + radius + 0.25, label_above,
                    ha='center', va='center',
                    color=C['dim'], fontsize=9, fontweight='bold')
        if label_below:
            ax.text(x, y - radius - 0.25, label_below,
                    ha='center', va='center',
                    color=C['muted'], fontsize=8)

    # ── 입력 뉴런 ──
    for i, iy in enumerate(in_ys):
        draw_neuron(layer_x['input'], iy, X[i], C['input'],
                    label_above=f'x{i+1}')

    # ── 합산 뉴런 (z) ──
    for j, oy in enumerate(out_ys):
        draw_neuron(layer_x['sum'], oy, z[j], C['sum'],
                    label_above=f'z{j+1} = Σ(xᵢwᵢ)+b')

        # 편향 표시
        ax.text(layer_x['sum'], oy - 0.65, f'+b{j+1}={b[j]:.1f}',
                ha='center', va='center', fontsize=8,
                color=C['bias'], fontweight='bold', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.15',
                         facecolor=C['bias']+'22', edgecolor=C['bias'],
                         linewidth=1.5))

    # ── 활성화 뉴런 ──
    for j, oy in enumerate(out_ys):
        # 활성화 함수 상자
        rect = FancyBboxPatch((layer_x['act'] - 0.4, oy - 0.4),
                               0.8, 0.8,
                               boxstyle="round,pad=0.1",
                               facecolor=C['act']+'22',
                               edgecolor=C['act'], linewidth=2, zorder=3)
        ax.add_patch(rect)
        ax.text(layer_x['act'], oy + 0.12, act_name,
                ha='center', va='center',
                color=C['act'], fontsize=9, fontweight='bold', zorder=5)
        ax.text(layer_x['act'], oy - 0.15, f'{a[j]:.3f}',
                ha='center', va='center',
                color=C['act'], fontsize=10, fontweight='bold',
                fontfamily='monospace', zorder=5)

    # ── 출력 뉴런 ──
    for j, oy in enumerate(out_ys):
        draw_neuron(layer_x['output'], oy, a[j], C['output'],
                    label_above=f'a{j+1} (출력)')

    # ── 층 제목 ──
    title_y = total_h + 0.3
    titles = [
        (layer_x['input'], '입력층\n(Input)', C['input']),
        (layer_x['sum'], '가중합\n(Weighted Sum)', C['sum']),
        (layer_x['act'], '활성화\n(Activation)', C['act']),
        (layer_x['output'], '출력층\n(Output)', C['output']),
    ]
    for tx, title, tc in titles:
        ax.text(tx, title_y, title, ha='center', va='center',
                color=tc, fontsize=11, fontweight='bold')

    # ── 수식 요약 (하단) ──
    formula_y = -0.5
    formula = f'z = X @ W + b = [{", ".join(f"{v:.2f}" for v in z)}]'
    formula2 = f'a = {act_name}(z) = [{", ".join(f"{v:.3f}" for v in a)}]'

    ax.text(5.25, formula_y, formula, ha='center', va='center',
            color=C['accent'], fontsize=11, fontfamily='monospace',
            fontweight='bold')
    ax.text(5.25, formula_y - 0.5, formula2, ha='center', va='center',
            color=C['act'], fontsize=11, fontfamily='monospace',
            fontweight='bold')

    plt.title('신경망 한 층의 순전파(Forward Pass) — 전체 구조',
              color=C['text'], fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

    return z, a

# ── 초기값으로 실행 ──
X = np.array([1.0, 0.5, -0.3])
W = np.array([[0.4, -0.2],
              [0.6,  0.8],
              [-0.5, 0.3]])
b = np.array([0.1, -0.1])

z, a = draw_network_layer(X, W, b, activation='relu')


---

## 4. 행렬 곱셈 — 단계별 계산 과정 시각화

`z = X @ W + b`가 내부적으로 어떻게 계산되는지 한 칸씩 보여준다.


In [ ]:
# ═══════════════════════════════════════════════════
# 행렬 곱셈 과정을 한 단계씩 시각화
# ═══════════════════════════════════════════════════

def visualize_matmul_steps(X, W, b):
    """X @ W + b 의 계산 과정을 요소별로 시각화한다."""

    n_in = len(X)
    n_out = W.shape[1]

    fig, axes = plt.subplots(1, n_out, figsize=(7 * n_out, 6),
                              facecolor=C['bg'])
    if n_out == 1:
        axes = [axes]

    for j in range(n_out):
        ax = axes[j]
        ax.set_facecolor(C['card'])
        ax.axis('off')
        ax.set_xlim(0, 10)
        ax.set_ylim(0, n_in * 2.2 + 3)

        ax.set_title(f'출력 뉴런 z{j+1} 계산 과정',
                     color=C['sum'], fontsize=14, fontweight='bold', pad=12)

        total = 0
        for i in range(n_in):
            y_pos = (n_in - i) * 2.2

            # x_i 값
            ax.text(0.8, y_pos, f'x{i+1}={X[i]:.1f}',
                    ha='center', va='center',
                    color=C['input'], fontsize=12, fontweight='bold',
                    fontfamily='monospace')

            # × 기호
            ax.text(2, y_pos, '×', ha='center', va='center',
                    color=C['dim'], fontsize=16, fontweight='bold')

            # w_ij 값
            ax.text(3.2, y_pos, f'w{i+1}{j+1}={W[i,j]:.1f}',
                    ha='center', va='center',
                    color=C['weight'], fontsize=12, fontweight='bold',
                    fontfamily='monospace')

            # = 기호
            ax.text(4.8, y_pos, '=', ha='center', va='center',
                    color=C['dim'], fontsize=16, fontweight='bold')

            # 결과
            product = X[i] * W[i, j]
            total += product
            ax.text(6, y_pos, f'{product:+.2f}',
                    ha='center', va='center',
                    color=C['accent'], fontsize=13, fontweight='bold',
                    fontfamily='monospace')

            # 누적 막대
            bar_width = abs(product) * 1.2
            bar_color = C['act'] if product >= 0 else C['sum']
            ax.barh(y_pos, bar_width, height=0.4,
                    left=7, color=bar_color, alpha=0.4)

        # 구분선
        y_sum = 0.8
        ax.plot([0.2, 9.5], [n_in * 2.2 - (n_in) * 2.2 + 1.5,
                              n_in * 2.2 - (n_in) * 2.2 + 1.5],
                color=C['muted'], linewidth=1, alpha=0.5)

        # 합산
        ax.text(0.3, y_sum + 0.7, 'Σ(합산)', ha='left', va='center',
                color=C['dim'], fontsize=10)
        ax.text(6, y_sum + 0.7, f'{total:+.2f}', ha='center', va='center',
                color=C['accent'], fontsize=13, fontweight='bold',
                fontfamily='monospace')

        # + 편향
        ax.text(0.3, y_sum, f'+ b{j+1}', ha='left', va='center',
                color=C['bias'], fontsize=11, fontweight='bold')
        ax.text(6, y_sum, f'{b[j]:+.2f}', ha='center', va='center',
                color=C['bias'], fontsize=13, fontweight='bold',
                fontfamily='monospace')

        # 최종 결과
        final = total + b[j]
        box = FancyBboxPatch((4.5, y_sum - 1), 3, 0.7,
                              boxstyle="round,pad=0.15",
                              facecolor=C['sum']+'33',
                              edgecolor=C['sum'], linewidth=2)
        ax.add_patch(box)
        ax.text(6, y_sum - 0.65, f'z{j+1} = {final:.2f}',
                ha='center', va='center',
                color=C['sum'], fontsize=14, fontweight='bold',
                fontfamily='monospace')

    plt.suptitle('행렬 곱셈 X @ W + b — 요소별 계산 과정',
                 color=C['text'], fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# 실행
X = np.array([1.0, 0.5, -0.3])
W = np.array([[0.4, -0.2],
              [0.6,  0.8],
              [-0.5, 0.3]])
b = np.array([0.1, -0.1])

visualize_matmul_steps(X, W, b)

print("\n🔑 핵심: NumPy의 @ 연산자는 이 모든 곱셈과 덧셈을 '한 번에' 수행한다.")
print("   for문으로 하나씩 계산하면 느리다. 행렬 연산으로 묶으면 GPU가 병렬 처리한다.")


---

## 5. ⚡ 인터랙티브 시각화 — 직접 조작해 보자!

슬라이더를 움직여서 입력값, 가중치, 편향을 바꾸면 **순전파 결과가 실시간으로 변한다.**

> 🎯 **실험 과제:**
> 1. 입력 x₁을 -2에서 2까지 움직여 보라. 출력이 어떻게 변하는가?
> 2. 가중치 w₁₁을 0으로 만들면? → x₁이 출력에 영향을 주지 않게 된다!
> 3. ReLU → Sigmoid로 바꾸면 출력 범위가 어떻게 달라지는가?


In [ ]:
# ═══════════════════════════════════════════════════
# 인터랙티브 위젯 — 실시간 순전파 시각화
# ═══════════════════════════════════════════════════

# 출력 영역
output_area = widgets.Output()

def interactive_forward_pass(x1, x2, x3, w11, w12, w21, w22, w31, w32,
                              b1, b2, activation):
    """슬라이더 값을 받아 순전파를 계산하고 시각화한다."""

    X = np.array([x1, x2, x3])
    W = np.array([[w11, w12], [w21, w22], [w31, w32]])
    b = np.array([b1, b2])

    z = X @ W + b
    if activation == 'ReLU':
        a = np.maximum(0, z)
        act_fn = 'ReLU'
    else:
        a = 1 / (1 + np.exp(-z))
        act_fn = 'Sigmoid'

    with output_area:
        clear_output(wait=True)

        fig = plt.figure(figsize=(16, 7), facecolor=C['bg'])

        # ── 왼쪽: 네트워크 그래프 ──
        ax1 = fig.add_axes([0.02, 0.05, 0.48, 0.85])
        ax1.set_facecolor(C['bg'])
        ax1.axis('off')
        ax1.set_xlim(-0.5, 10)
        ax1.set_ylim(-0.5, 6)

        # 좌표
        in_xs, in_ys = [1]*3, [5, 3, 1]
        out_xs, out_ys = [4.5]*2, [4, 2]
        act_xs, act_ys = [7]*2, [4, 2]
        final_xs, final_ys = [9]*2, [4, 2]

        # 연결선 + 가중치
        for i in range(3):
            for j in range(2):
                w_val = W[i, j]
                lw = max(0.5, min(4, abs(w_val) * 3))
                alpha_v = max(0.15, min(0.7, abs(w_val) * 0.7))
                lc = C['weight'] if w_val >= 0 else C['sum']

                ax1.plot([in_xs[i]+0.4, out_xs[j]-0.4],
                         [in_ys[i], out_ys[j]],
                         color=lc, lw=lw, alpha=alpha_v, zorder=1)

                # 가중치 텍스트
                mx = (in_xs[i] + out_xs[j]) / 2 + (j-0.5)*0.3
                my = (in_ys[i] + out_ys[j]) / 2 + 0.15
                ax1.text(mx, my, f'{w_val:.1f}', ha='center', va='center',
                         fontsize=7, color=C['weight'], fontfamily='monospace',
                         alpha=0.85,
                         bbox=dict(facecolor=C['bg'], edgecolor=C['weight'],
                                  boxstyle='round,pad=0.1', alpha=0.6, lw=0.5))

        # 활성화 연결
        for j in range(2):
            ax1.annotate('', xy=(act_xs[j]-0.35, act_ys[j]),
                        xytext=(out_xs[j]+0.4, out_ys[j]),
                        arrowprops=dict(arrowstyle='->', color=C['act'],
                                       lw=1.5, alpha=0.5))
            ax1.annotate('', xy=(final_xs[j]-0.35, final_ys[j]),
                        xytext=(act_xs[j]+0.35, act_ys[j]),
                        arrowprops=dict(arrowstyle='->', color=C['output'],
                                       lw=2, alpha=0.6))

        # 뉴런 그리기
        def circ(x, y, val, color, label='', r=0.35, fs=12):
            ax1.add_patch(plt.Circle((x,y), r, color=color, alpha=0.15, zorder=3))
            ax1.add_patch(plt.Circle((x,y), r, fill=False,
                                      edgecolor=color, lw=2.5, zorder=4))
            ax1.text(x, y, f'{val:.2f}', ha='center', va='center',
                     color=color, fontsize=fs, fontweight='bold',
                     fontfamily='monospace', zorder=5)
            if label:
                ax1.text(x, y+r+0.2, label, ha='center', va='center',
                         color=C['dim'], fontsize=9, fontweight='bold')

        # 입력
        for i in range(3):
            circ(in_xs[i], in_ys[i], X[i], C['input'], f'x{i+1}')

        # 합산
        for j in range(2):
            circ(out_xs[j], out_ys[j], z[j], C['sum'], f'z{j+1}')
            ax1.text(out_xs[j], out_ys[j]-0.55,
                     f'+b={b[j]:.1f}', ha='center', va='center',
                     fontsize=7, color=C['bias'], fontweight='bold',
                     fontfamily='monospace')

        # 활성화
        for j in range(2):
            rect = FancyBboxPatch((act_xs[j]-0.3, act_ys[j]-0.3),
                                   0.6, 0.6, boxstyle="round,pad=0.08",
                                   facecolor=C['act']+'22',
                                   edgecolor=C['act'], lw=2, zorder=3)
            ax1.add_patch(rect)
            ax1.text(act_xs[j], act_ys[j]+0.1, act_fn, ha='center',
                     va='center', color=C['act'], fontsize=8,
                     fontweight='bold', zorder=5)
            ax1.text(act_xs[j], act_ys[j]-0.12, f'{a[j]:.3f}',
                     ha='center', va='center', color=C['act'],
                     fontsize=9, fontweight='bold', fontfamily='monospace',
                     zorder=5)

        # 출력
        for j in range(2):
            circ(final_xs[j], final_ys[j], a[j], C['output'], f'a{j+1}(출력)')

        # 제목
        ax1.set_title('네트워크 구조', color=C['text'],
                      fontsize=13, fontweight='bold')

        # ── 오른쪽: 계산 과정 ──
        ax2 = fig.add_axes([0.55, 0.05, 0.42, 0.85])
        ax2.set_facecolor(C['card'])
        ax2.axis('off')
        ax2.set_xlim(0, 10)
        ax2.set_ylim(0, 10)

        ax2.set_title('계산 과정 (NumPy 코드)', color=C['accent'],
                      fontsize=13, fontweight='bold')

        lines = [
            ('import numpy as np', C['dim'], 9.5),
            ('', C['dim'], 9.1),
            (f'X = np.array([{x1:.1f}, {x2:.1f}, {x3:.1f}])', C['input'], 8.7),
            (f'W = np.array([[{w11:.1f}, {w12:.1f}],', C['weight'], 8.2),
            (f'              [{w21:.1f}, {w22:.1f}],', C['weight'], 7.8),
            (f'              [{w31:.1f}, {w32:.1f}]])', C['weight'], 7.4),
            (f'b = np.array([{b1:.1f}, {b2:.1f}])', C['bias'], 6.9),
            ('', C['dim'], 6.5),
            ('z = X @ W + b', C['accent'], 6.1),
            (f'  → z = [{z[0]:.3f}, {z[1]:.3f}]', C['sum'], 5.6),
            ('', C['dim'], 5.2),
        ]

        if activation == 'ReLU':
            lines.append(('a = np.maximum(0, z)  # ReLU', C['act'], 4.8))
        else:
            lines.append(('a = 1/(1+np.exp(-z))  # Sigmoid', C['act'], 4.8))

        lines.append((f'  → a = [{a[0]:.4f}, {a[1]:.4f}]', C['act'], 4.3))

        for text, color, y in lines:
            ax2.text(0.3, y, text, ha='left', va='center',
                     color=color, fontsize=10, fontfamily='monospace',
                     fontweight='bold' if '→' in text else 'normal')

        # 계산 상세 (출력 뉴런 1)
        ax2.plot([0.3, 9.7], [3.5, 3.5], color=C['muted'], lw=0.5, alpha=0.5)
        ax2.text(0.3, 3.1, f'▼ z₁ 계산 상세:', color=C['dim'], fontsize=9)

        detail_parts = []
        for i in range(3):
            sign = '+' if i > 0 and X[i]*W[i,0] >= 0 else ''
            if i == 0 and X[i]*W[i,0] < 0:
                sign = ''
            detail_parts.append(f'{X[i]:.1f}×{W[i,0]:.1f}')

        detail = ' + '.join(detail_parts) + f' + {b[0]:.1f}'
        products = [X[i]*W[i,0] for i in range(3)]
        detail2 = ' + '.join([f'{p:.2f}' for p in products]) + f' + {b[0]:.1f}'

        ax2.text(0.5, 2.6, f'z₁ = {detail}', color=C['sum'],
                 fontsize=9, fontfamily='monospace')
        ax2.text(0.5, 2.1, f'   = {detail2}', color=C['sum'],
                 fontsize=9, fontfamily='monospace')
        ax2.text(0.5, 1.6, f'   = {z[0]:.4f}', color=C['sum'],
                 fontsize=10, fontweight='bold', fontfamily='monospace')

        # 파라미터 수 표시
        n_params = W.size + b.size
        ax2.text(0.3, 0.5, f'총 파라미터 수: {W.size}(가중치) + {b.size}(편향) = {n_params}개',
                 color=C['accent'], fontsize=10, fontweight='bold')

        plt.show()

# ── 위젯 생성 ──
style = {'description_width': '50px'}
layout = widgets.Layout(width='280px')
layout_short = widgets.Layout(width='200px')

# 입력 슬라이더
input_sliders = [
    widgets.FloatSlider(value=1.0, min=-2, max=2, step=0.1,
                        description=f'x{i+1}:', style=style, layout=layout,
                        readout_format='.1f')
    for i in range(3)
]

# 가중치 슬라이더
weight_sliders = []
w_init = [[0.4, -0.2], [0.6, 0.8], [-0.5, 0.3]]
for i in range(3):
    for j in range(2):
        weight_sliders.append(
            widgets.FloatSlider(value=w_init[i][j], min=-2, max=2, step=0.1,
                                description=f'w{i+1}{j+1}:', style=style,
                                layout=layout_short, readout_format='.1f')
        )

# 편향 슬라이더
bias_sliders = [
    widgets.FloatSlider(value=v, min=-2, max=2, step=0.1,
                        description=f'b{i+1}:', style=style, layout=layout_short,
                        readout_format='.1f')
    for i, v in enumerate([0.1, -0.1])
]

# 활성화 함수 선택
act_toggle = widgets.ToggleButtons(
    options=['ReLU', 'Sigmoid'],
    value='ReLU',
    description='활성화:',
    style=style,
    button_style='info'
)

# 레이아웃 구성
input_box = widgets.VBox([
    widgets.HTML('<b style="color:#3b82f6;font-size:13px">🔵 입력(Input)</b>'),
    *input_sliders
])

weight_box = widgets.VBox([
    widgets.HTML('<b style="color:#f59e0b;font-size:13px">🟡 가중치(Weight)</b>'),
    widgets.HBox([
        widgets.VBox(weight_sliders[0:2]),
        widgets.VBox(weight_sliders[2:4]),
        widgets.VBox(weight_sliders[4:6]),
    ])
])

bias_act_box = widgets.VBox([
    widgets.HTML('<b style="color:#8b5cf6;font-size:13px">🟣 편향(Bias) + 활성화</b>'),
    *bias_sliders,
    act_toggle,
])

controls = widgets.HBox([input_box, weight_box, bias_act_box],
                          layout=widgets.Layout(gap='20px'))

# 연결
def on_change(*args):
    interactive_forward_pass(
        input_sliders[0].value, input_sliders[1].value, input_sliders[2].value,
        weight_sliders[0].value, weight_sliders[1].value,
        weight_sliders[2].value, weight_sliders[3].value,
        weight_sliders[4].value, weight_sliders[5].value,
        bias_sliders[0].value, bias_sliders[1].value,
        act_toggle.value,
    )

for s in input_sliders + weight_sliders + bias_sliders:
    s.observe(on_change, 'value')
act_toggle.observe(on_change, 'value')

# 표시
display(widgets.HTML('''
<div style="background:#1e293b;padding:12px 18px;border-radius:10px;
            border:1px solid #334155;margin-bottom:10px">
  <h3 style="color:#e2e8f0;margin:0 0 5px">⚡ 인터랙티브 순전파 시각화</h3>
  <p style="color:#94a3b8;margin:0;font-size:13px">
    슬라이더를 움직여 입력·가중치·편향을 조절하면 순전파 결과가 실시간으로 변한다.</p>
</div>
'''))
display(controls)
display(output_area)

# 초기 실행
on_change()


---

## 6. 🧪 실험 과제 — 직접 해 보자!

위 인터랙티브 시각화에서 다음 실험을 수행하라.

| # | 실험 | 관찰 포인트 |
|:---:|---|---|
| 1 | x₁을 -2 → +2로 천천히 이동 | 출력 a₁, a₂가 어떻게 변하는가? |
| 2 | w₁₁을 0으로 설정 | x₁이 z₁에 미치는 영향이 사라지는가? |
| 3 | 모든 가중치를 0으로 설정 | 출력이 편향(b)만 남는가? |
| 4 | ReLU → Sigmoid 전환 | 출력 범위가 어떻게 달라지는가? |
| 5 | 편향 b₁을 -5로 설정하고 ReLU 사용 | z₁이 음수이면 ReLU 출력은? |
| 6 | 가중치를 모두 양수(+1)로 설정 | 입력이 모두 양수이면 출력은 항상 양수인가? |

> **📝 관찰을 반드시 수치로 기록하라.** "출력이 커졌다"가 아니라 "a₁이 0.42에서 1.37로 증가했다"로 작성한다.


---

## 7. 🚀 이것이 최신 AI와 어떻게 연결되는가?

방금 실습한 `X @ W + b` → 활성화 — 이것이 **모든 신경망의 기본 단위**이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 규모 비교: 우리의 1층 vs 실제 AI 모델
# ═══════════════════════════════════════════════════

models = {
    '지금 만든 것':       {'params': 8,           'layers': 1,
                          'input': 3, 'output': 2,
                          'year': 2024, 'color': C['input']},
    'LeNet-5\n(1998)':   {'params': 60_000,      'layers': 5,
                          'input': 784, 'output': 10,
                          'year': 1998, 'color': C['bias']},
    'AlexNet\n(2012)':   {'params': 61_000_000,  'layers': 8,
                          'input': 150528, 'output': 1000,
                          'year': 2012, 'color': C['weight']},
    'BERT\n(2018)':      {'params': 340_000_000, 'layers': 24,
                          'input': 768, 'output': 768,
                          'year': 2018, 'color': C['sum']},
    'GPT-3\n(2020)':     {'params': 175_000_000_000, 'layers': 96,
                          'input': 12288, 'output': 12288,
                          'year': 2020, 'color': C['act']},
    'GPT-4\n(2023, 추정)': {'params': 1_800_000_000_000, 'layers': 120,
                           'input': 16384, 'output': 16384,
                           'year': 2023, 'color': C['accent']},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor=C['bg'])

# ── 왼쪽: 파라미터 수 비교 (로그 스케일) ──
ax1.set_facecolor(C['card'])
names = list(models.keys())
params = [m['params'] for m in models.values()]
colors = [m['color'] for m in models.values()]

bars = ax1.barh(range(len(names)), [np.log10(p) for p in params],
                color=colors, alpha=0.7, height=0.6, edgecolor='white',
                linewidth=0.5)

ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, color=C['text'], fontsize=10, fontweight='bold')
ax1.set_xlabel('파라미터 수 (10의 거듭제곱)', color=C['dim'], fontsize=11)
ax1.set_title('모델별 파라미터 수 비교 (로그 스케일)',
              color=C['text'], fontsize=14, fontweight='bold')

# 값 라벨
for i, (bar, p) in enumerate(zip(bars, params)):
    if p >= 1_000_000_000:
        label = f'{p/1_000_000_000:.0f}B'
    elif p >= 1_000_000:
        label = f'{p/1_000_000:.0f}M'
    elif p >= 1_000:
        label = f'{p/1_000:.0f}K'
    else:
        label = str(p)
    ax1.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
             label, va='center', color=colors[i], fontsize=10,
             fontweight='bold', fontfamily='monospace')

ax1.tick_params(colors=C['muted'])
for spine in ax1.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 오른쪽: 핵심 메시지 ──
ax2.set_facecolor(C['card'])
ax2.axis('off')

messages = [
    ("원리는 완전히 동일하다", C['accent'], 18),
    ("", None, 10),
    ("z = X @ W + b", C['weight'], 22),
    ("a = activation(z)", C['act'], 22),
    ("", None, 10),
    ("이것을 수백 번 반복하고,", C['text'], 14),
    ("가중치(W)를 데이터로부터", C['text'], 14),
    ("자동으로 학습하는 것 —", C['text'], 14),
    ("", None, 6),
    ("그것이 딥러닝이다.", C['accent'], 18),
    ("", None, 12),
    ("파라미터 8개 → 1조 8000억 개", C['sum'], 13),
    ("규모만 다를 뿐, 수학은 같다.", C['dim'], 12),
]

y_pos = 0.92
for text, color, size in messages:
    if text:
        ax2.text(0.5, y_pos, text, transform=ax2.transAxes,
                 ha='center', va='center', color=color,
                 fontsize=size, fontweight='bold',
                 fontfamily='monospace' if '@' in text or '=' in text or '(' in text else None)
    y_pos -= size * 0.0035 + 0.01

plt.tight_layout()
plt.show()


---

## 8. 🎬 애니메이션 — 순전파가 흐르는 과정

단계별로 신호가 전달되는 과정을 애니메이션으로 본다.


In [ ]:
# ═══════════════════════════════════════════════════
# 순전파 단계별 애니메이션
# ═══════════════════════════════════════════════════
from matplotlib.animation import FuncAnimation
from IPython.display import HTML as IPyHTML

X = np.array([1.0, 0.5, -0.3])
W = np.array([[0.4, -0.2], [0.6, 0.8], [-0.5, 0.3]])
b = np.array([0.1, -0.1])
z = X @ W + b
a = np.maximum(0, z)  # ReLU

fig, ax = plt.subplots(figsize=(14, 7), facecolor=C['bg'])
ax.set_facecolor(C['bg'])
ax.set_xlim(-1, 12)
ax.set_ylim(-1.5, 7)
ax.axis('off')

# 좌표
in_pos = [(1, 5), (1, 3), (1, 1)]
sum_pos = [(5, 4), (5, 2)]
act_pos = [(8, 4), (8, 2)]
out_pos = [(10.5, 4), (10.5, 2)]

step_titles = [
    "단계 0: 초기 상태",
    "단계 1: 입력값 표시",
    "단계 2: 가중치 연결 (X × W)",
    "단계 3: 가중합 + 편향 (z = X@W + b)",
    "단계 4: 활성화 함수 (ReLU)",
    "단계 5: 출력 완성!",
]

title_text = ax.text(6, 6.5, '', ha='center', va='center',
                      color=C['text'], fontsize=16, fontweight='bold')
formula_text = ax.text(6, -1, '', ha='center', va='center',
                        color=C['accent'], fontsize=12,
                        fontfamily='monospace', fontweight='bold')

def draw_circle(cx, cy, val, color, alpha=1.0, r=0.4, fs=12, label=''):
    circle_bg = plt.Circle((cx, cy), r, color=color, alpha=alpha*0.2)
    circle_bd = plt.Circle((cx, cy), r, fill=False,
                            edgecolor=color, lw=2.5, alpha=alpha)
    ax.add_patch(circle_bg)
    ax.add_patch(circle_bd)
    ax.text(cx, cy, f'{val:.2f}', ha='center', va='center',
            color=color, fontsize=fs, fontweight='bold',
            fontfamily='monospace', alpha=alpha)
    if label:
        ax.text(cx, cy+r+0.25, label, ha='center', va='center',
                color=C['dim'], fontsize=9, alpha=alpha)
    return [circle_bg, circle_bd]

def animate(frame):
    ax.clear()
    ax.set_facecolor(C['bg'])
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1.5, 7)
    ax.axis('off')

    step = frame % 6

    # 제목
    ax.text(6, 6.5, step_titles[step], ha='center', va='center',
            color=C['text'], fontsize=16, fontweight='bold')

    # 단계별 그리기
    if step >= 1:  # 입력
        for i, (cx, cy) in enumerate(in_pos):
            draw_circle(cx, cy, X[i], C['input'], label=f'x{i+1}')

    if step >= 2:  # 연결선
        for i, (ix, iy) in enumerate(in_pos):
            for j, (sx, sy) in enumerate(sum_pos):
                w_val = W[i, j]
                lw = max(0.5, min(3, abs(w_val) * 2.5))
                lc = C['weight'] if w_val >= 0 else C['sum']
                ax.plot([ix+0.4, sx-0.4], [iy, sy],
                        color=lc, lw=lw, alpha=0.5)
                mx, my = (ix+sx)/2, (iy+sy)/2
                ax.text(mx+0.1, my+0.15, f'{w_val:.1f}',
                        ha='center', va='center', fontsize=7,
                        color=C['weight'], fontfamily='monospace',
                        bbox=dict(facecolor=C['bg'], edgecolor=C['weight'],
                                 boxstyle='round,pad=0.1', alpha=0.6, lw=0.5))

    if step >= 3:  # 합산 뉴런
        for j, (sx, sy) in enumerate(sum_pos):
            draw_circle(sx, sy, z[j], C['sum'], label=f'z{j+1}')
            ax.text(sx, sy-0.6, f'+b={b[j]:.1f}',
                    ha='center', va='center', fontsize=8,
                    color=C['bias'], fontweight='bold', fontfamily='monospace')

    if step >= 4:  # 활성화
        for j, (sx, sy) in enumerate(sum_pos):
            ax.annotate('', xy=(act_pos[j][0]-0.35, act_pos[j][1]),
                        xytext=(sx+0.4, sy),
                        arrowprops=dict(arrowstyle='->', color=C['act'],
                                       lw=1.5))

        for j, (ax_x, ay) in enumerate(act_pos):
            rect = FancyBboxPatch((ax_x-0.35, ay-0.35), 0.7, 0.7,
                                   boxstyle="round,pad=0.08",
                                   facecolor=C['act']+'22',
                                   edgecolor=C['act'], lw=2)
            ax.add_patch(rect)
            ax.text(ax_x, ay+0.12, 'ReLU', ha='center', va='center',
                    color=C['act'], fontsize=9, fontweight='bold')
            ax.text(ax_x, ay-0.12, f'{a[j]:.3f}', ha='center', va='center',
                    color=C['act'], fontsize=10, fontweight='bold',
                    fontfamily='monospace')

    if step >= 5:  # 출력
        for j in range(2):
            ax.annotate('', xy=(out_pos[j][0]-0.35, out_pos[j][1]),
                        xytext=(act_pos[j][0]+0.35, act_pos[j][1]),
                        arrowprops=dict(arrowstyle='->', color=C['output'],
                                       lw=2))
            draw_circle(out_pos[j][0], out_pos[j][1], a[j],
                        C['output'], label=f'a{j+1}(출력)')

    # 수식
    formulas = [
        '',
        f'X = [{", ".join(f"{v:.1f}" for v in X)}]',
        'z = X @ W    (행렬 곱셈 진행 중...)',
        f'z = X @ W + b = [{z[0]:.2f}, {z[1]:.2f}]',
        f'a = ReLU(z) = [{a[0]:.3f}, {a[1]:.3f}]',
        f'✅ 순전파 완료!  a = [{a[0]:.3f}, {a[1]:.3f}]',
    ]
    ax.text(6, -1, formulas[step], ha='center', va='center',
            color=C['accent'], fontsize=12, fontfamily='monospace',
            fontweight='bold')

    return []

anim = FuncAnimation(fig, animate, frames=24, interval=1500, blit=False)
plt.close(fig)
IPyHTML(anim.to_jshtml())


---

## 9. for문 vs NumPy — 속도 비교

"왜 `for`문 대신 NumPy를 쓰는가?"에 대한 최종 답이다.


In [ ]:
# ═══════════════════════════════════════════════════
# for문 vs NumPy 행렬 곱 — 속도 비교
# ═══════════════════════════════════════════════════
import time

sizes = [10, 100, 1000, 5000]
for_times = []
np_times = []

for n in sizes:
    X_big = np.random.randn(n)
    W_big = np.random.randn(n, n)
    b_big = np.random.randn(n)

    # for문 버전
    start = time.perf_counter()
    z_for = np.zeros(n)
    for j in range(n):
        s = 0
        for i in range(n):
            s += X_big[i] * W_big[i, j]
        z_for[j] = s + b_big[j]
    t_for = time.perf_counter() - start
    for_times.append(t_for)

    # NumPy 버전
    start = time.perf_counter()
    for _ in range(100):  # 100번 반복하여 측정
        z_np = X_big @ W_big + b_big
    t_np = (time.perf_counter() - start) / 100
    np_times.append(t_np)

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=C['bg'])

# 시간 비교
ax1.set_facecolor(C['card'])
x_pos = np.arange(len(sizes))
w_bar = 0.35
ax1.bar(x_pos - w_bar/2, [t*1000 for t in for_times], w_bar,
        color=C['sum'], alpha=0.7, label='for문', edgecolor='white', lw=0.5)
ax1.bar(x_pos + w_bar/2, [t*1000 for t in np_times], w_bar,
        color=C['act'], alpha=0.7, label='NumPy @', edgecolor='white', lw=0.5)
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'{n}×{n}' for n in sizes], color=C['dim'])
ax1.set_ylabel('시간 (ms)', color=C['dim'])
ax1.set_title('실행 시간 비교', color=C['text'], fontsize=14, fontweight='bold')
ax1.legend(facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
ax1.set_yscale('log')
ax1.tick_params(colors=C['muted'])
for spine in ax1.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# 속도 배율
ax2.set_facecolor(C['card'])
speedups = [f/n for f, n in zip(for_times, np_times)]
bars = ax2.bar(x_pos, speedups, color=C['accent'], alpha=0.7,
               edgecolor='white', lw=0.5)
for bar, sp in zip(bars, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{sp:.0f}×', ha='center', va='bottom',
             color=C['accent'], fontsize=13, fontweight='bold',
             fontfamily='monospace')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'{n}×{n}' for n in sizes], color=C['dim'])
ax2.set_ylabel('NumPy가 몇 배 빠른가?', color=C['dim'])
ax2.set_title('속도 향상 배율', color=C['text'], fontsize=14, fontweight='bold')
ax2.tick_params(colors=C['muted'])
for spine in ax2.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

plt.tight_layout()
plt.show()

print(f"\n{'═'*60}")
print(f"  📊 결과 요약")
print(f"{'═'*60}")
for i, n in enumerate(sizes):
    print(f"  {n:>5}×{n} 행렬:  for문 {for_times[i]*1000:>10.2f}ms  |  "
          f"NumPy {np_times[i]*1000:>8.4f}ms  |  {speedups[i]:>6.0f}× 빠름")
print(f"{'═'*60}")
print(f"\n💡 GPU를 쓰면 이 차이가 수천~수만 배로 벌어진다.")
print(f"   GPT-4의 순전파가 실시간으로 가능한 이유가 바로 이것이다.")


---

## 🎯 핵심 정리

### 이 노트북에서 배운 것

| 개념 | 핵심 | NumPy 코드 |
|---|---|---|
| **뉴런**(Neuron) | 곱셈 → 덧셈 → 활성화 | `a = relu(x * w + b)` |
| **한 층**(Layer) | 행렬 곱으로 한 번에 계산 | `a = relu(X @ W + b)` |
| **활성화 함수** | 비선형성을 부여하여 복잡한 패턴 학습 | `np.maximum(0, z)` |
| **행렬 곱** | for문 없이 병렬 연산 → GPU 가속의 본질 | `X @ W` |

### 최신 AI와의 연결

| 우리가 배운 것 | 최신 AI에서의 확장 |
|---|---|
| `X @ W + b` | **모든 신경망의 기본 연산** — GPT, BERT, Stable Diffusion 포함 |
| ReLU / Sigmoid | GELU(GPT), SiLU/Swish(LLaMA) 등 변형 사용 |
| 파라미터 8개 | GPT-4는 **~1조 8000억 개** — 규모만 다르고 원리는 동일 |
| NumPy `@` 연산자 | PyTorch `torch.matmul()` — 문법만 다르고 동작은 같다 |
| for문 vs 행렬 곱 | GPU 병렬 처리 — 이것이 딥러닝을 가능하게 만든 핵심 기술 |

### 다음 단계

```
4주차 기본      →   6주차 ML 워크플로우     →   10주차 신경망 심화
X @ W + b           train_test_split           역전파(Backpropagation)
(순전파 이해)        (데이터 분리)               (가중치 자동 학습)
```

> 💬 **"지금은 가중치를 우리가 직접 슬라이더로 조절했다. 하지만 이 가중치를 데이터로부터 자동으로 찾는 방법이 있다. 그것이 바로 '학습(Training)'이며, 10주차에서 다룬다."**
